# Mean-Variance Portfolio Selection Mean-Field Control Benchmark

Reference: `files/reference/continuous_benchmarks.tex`, Sec. "Mean-Variance Portfolio Selection". Like `lq.ipynb`, **the perturbed objective $J^\lambda(\theta)$, its exact policy gradient, and the unperturbed optimal policy are all available in closed form** — here even though the terminal wealth law is generally non-Gaussian (only its first two moments are used, and they close exactly).

**Model.** Wealth $X_t\in\mathbb R$, monetary amount $\alpha_t\in\mathbb R$ invested in the risky asset (the rest sits in the risk-free asset). $X_{t+1}=s_tX_t+R_{t+1}\alpha_t$, with $R_{t+1}$ the risky asset's excess return (independent across $t$; only its first two moments $\bar r_t,\sigma_{R,t}^2$ matter for the exact objective/gradient — the return *law* itself is a runtime choice, see the robustness section below). There is **no running reward**; the terminal reward is $g(x,m)=x-\chi(x-\bar m)^2$, so
$$J^0(\theta) = \mathbb E[X_T] - \chi\,\mathrm{Var}(X_T)$$
the precommitment mean-variance criterion (a **reward to maximize**, unlike `lq.ipynb`'s cost-to-minimize convention — see `mfc.algorithms.portfolio`'s module docstring).

**Policy.** $\pi_t^\theta(\cdot\mid x,m)=\mathcal N(k_t(x-\bar m)+\ell_t,\tau_t^2)$, $\theta=((k_t,\ell_t))_{t=0}^{T-1}$ — the same genuinely time-indexed $(T,2)$ parametrization as LQ.

**Baseline parameters** (reference "Training and evaluation", used directly as `PortfolioConfig`'s defaults — unlike LQ, this benchmark's reference gives a full numeric baseline case, not just formulas): $T=10$, $X_0\sim\mathcal N(1,0.04)$, $s_t=1$, $\bar r_t=0.02$, $\sigma_{R,t}=0.08$, $\chi=10$, $\tau_t=0.02$, $\rho=1$. $\theta$ is initialized at $k_t=\ell_t=0$ (the reference's own choice).

**Two training algorithms**, mirroring LQ exactly: **`exact_gradient`** (the closed-form $O(T)$ adjoint gradient, no sampling) and **`reinforce`** (the classical-REINFORCE ablation, missing the mean-field sensitivity term). Since $\tau=0.02$ is tiny, the REINFORCE policy score (magnitude $\propto 1/\tau^2$) is very high-variance here — `reinforce` needs a much larger Monte Carlo batch ($B=2000$) than this repo's usual 200 to make visible progress at all; this is itself a real finding, not a tuning failure (see `configs/portfolio.py`'s module docstring).

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch

torch.set_default_dtype(torch.float64)
torch.set_default_device("cpu")  # small scalar/(T,B) ops: GPU kernel-launch overhead dominates here,
                                  # same finding as this repo's twostate/LQ profiling

from configs.portfolio import MID
from mfc.environments.portfolio import Portfolio, PortfolioConfig
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import apply_style, color_for, new_figure, style_legend
from scripts.train import run_continuous
from scripts.test import load_runs

## Configuration and budget

This notebook demonstrates the **mid** run tier from `configs/portfolio.py`: one seed, the reference horizon $T=10$, and the full training length. Run `scripts/train.py --env portfolio --alg exact_gradient --config main` (and `--alg reinforce`) separately for the full main-tier sweep (5 seeds, horizons $T\in\{5,10,20\}$, used by the horizon-scaling cell below when available).

In [ ]:
cfg = MID
env = Portfolio(device="cpu")  # matches the default device set above and scripts.train.run_continuous's own choice
T = cfg.horizons[0]

print(f"algorithms:  {cfg.algorithms}")
print(f"lambdas:     {cfg.lambdas}  (the reference's own grid)")
print(f"T={T}, seeds={cfg.seeds}, lr={cfg.lr}, n_train={cfg.n_train}, B(reinforce)={cfg.B}")
print(f"market: s={env.config.s}, r_bar={env.config.r_bar}, sigma_R={env.config.sigma_R}")
print(f"chi={env.config.chi}, tau={env.config.tau}, rho={env.config.rho}, mu0={env.config.mu0}, Sigma0={env.config.Sigma0}")

## Ground truth: the optimal policy

The unperturbed ($\lambda=0$) optimal $\theta^\star$ (`Portfolio.optimal_theta`) and its value $J^0(\theta^\star)$ — exact, closed-form, no training needed. Every plot below compares the learned policy against this. $k_t^\star$ is constant across $t$ (time-homogeneous baseline config); $\ell_t^\star$ decays geometrically moving away from the terminal time.

In [ ]:
theta_star = env.optimal_theta(T)
J_star = env.exact_objective(theta_star, 0.0).item()
grad_at_star = env.exact_gradient(theta_star, 0.0)

print("theta* (k_t, l_t):")
print(theta_star)
print(f"J^0(theta*) = {J_star:.6f}")
print(f"||grad J^0(theta*)|| = {grad_at_star.norm().item():.2e}  (should be ~0: theta* is a stationary point)")

mu_star, Sigma_star = env.forward_moments(theta_star, 0.0)
print(f"E[X_t] under theta*: {mu_star.tolist()}")

## Train (or load cached results)

Loads every saved run under `runs/portfolio/mid/` for both algorithms; trains first if none exist yet.

In [ ]:
runs_dir = ROOT / "runs" / "portfolio" / "mid"

runs = []
for alg in cfg.algorithms:
    if not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_continuous("portfolio", alg, "mid")
    runs += load_runs("portfolio", alg, "mid")

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded; total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")
for r in sorted(runs, key=lambda r: (r["alg"], r["lam"])):
    print(f"  {r['alg']:<14} lambda={r['lam']:<6} seed={r['seed']}  elapsed={r['elapsed_seconds']:.1f}s  final validation J={r['validation_J'][-1].item():.4f}")

## Evolution of the validation objective

The exact validation objective $J^0(\theta_m)$ (closed-form, not Monte Carlo) every `validate_every` training iterations, one line per $(\text{algorithm},\lambda)$, against the optimal reference line. `exact_gradient` should converge smoothly and essentially exactly; `reinforce` is expected to be slow and noisy given the tiny $\tau=0.02$ (see the model markdown above).

In [ ]:
fig, ax = viz.plot_validation_curve(runs, optimal_J=J_star)
ax.set_title("Validation objective during training (dashed = optimum)", loc="left")

## Learned theta vs optimal theta

$k_t$ and $\ell_t$ across $t$, for $\lambda=0.1$'s learned policy under each algorithm, against $\theta^\star$ (dashed).

In [ ]:
by_alg_lambda = {(r["alg"], r["lam"]): r for r in runs}
theta_exact = by_alg_lambda[("exact_gradient", 0.1)]["theta_final"]
theta_reinforce = by_alg_lambda[("reinforce", 0.1)]["theta_final"]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
viz.plot_lq_theta(theta_exact, optimal_theta=theta_star, ax=axes[0])
axes[0].set_title("exact_gradient (lambda=0.1)", loc="left")
viz.plot_lq_theta(theta_reinforce, optimal_theta=theta_star, ax=axes[1])
axes[1].set_title("reinforce (lambda=0.1)", loc="left")
fig.tight_layout()

## Wealth distribution over time: learned vs optimal

$\mathbb E[X_t]\pm\mathrm{std}(X_t)$ under the learned policy, against the optimal trajectory. Wealth compounds forward, so both the mean and the spread should grow with $t$.

In [ ]:
mu_learned, Sigma_learned = env.forward_moments(theta_exact, 0.0)
fig, ax = viz.plot_gaussian_flow(mu_learned, Sigma_learned, optimal_mu=mu_star, optimal_Sigma=Sigma_star, label="learned (exact_gradient, lambda=0.1)")
ax.set_ylabel("X_t (mean ± 1 std)")
ax.set_title("Wealth over time: learned vs optimal", loc="left")

## $J^\lambda$ vs $J^0$, at the learned and optimal theta

Both sides are exact closed-form evaluations (`Portfolio.exact_objective`), not Monte Carlo estimates. `tests/test_portfolio.py` checks the reference's own claim that the perturbation bias scales *exactly* as $\lambda^2$ (verified to 6 decimal places at $\theta^\star$).

In [ ]:
lambdas = cfg.lambdas
J_lambda_at_learned = {lam: env.exact_objective(theta_exact, lam).item() for lam in lambdas}
J_lambda_at_optimal = {lam: env.exact_objective(theta_star, lam).item() for lam in lambdas}

fig, ax = new_figure()
ax.plot(lambdas, list(J_lambda_at_learned.values()), color=color_for(0), linewidth=2, marker="o", markersize=6, label="J^lambda(theta_learned)")
ax.plot(lambdas, list(J_lambda_at_optimal.values()), color=color_for(1), linewidth=2, marker="s", markersize=6, label="J^lambda(theta*)")
ax.axhline(J_star, color="black", linestyle="--", linewidth=1.5, label="J^0(theta*)")
apply_style(ax, xlabel="lambda", ylabel="J^lambda")
style_legend(ax)
ax.set_title("Perturbed objective vs lambda, at learned and optimal theta", loc="left")

## Optimality gap

The reference's own diagnostic 2 ("we train the policy from the zero initialization and compare the learned parameters with the exact solution... as well as the optimality gap $J^0(\theta^\star)-J^0(\theta)$"), for `exact_gradient` (trained at $\lambda=0.1$) across seeds.

In [ ]:
exact_runs_lam = [r for r in runs if r["alg"] == "exact_gradient" and r["lam"] == 0.1]
gaps = torch.tensor([J_star - env.exact_objective(r["theta_final"], 0.0).item() for r in exact_runs_lam])
theta_errs = torch.tensor([(r["theta_final"] - theta_star).norm().item() for r in exact_runs_lam])
print(f"optimality gap J^0(theta*) - J^0(theta_learned): mean={gaps.mean().item():.6f}  std={gaps.std().item() if len(gaps) > 1 else 0.0:.6f}")
print(f"||theta_learned - theta*||: mean={theta_errs.mean().item():.6f}  std={theta_errs.std().item() if len(theta_errs) > 1 else 0.0:.6f}")

## Gradient bias/variance vs the exact oracle, across lambda

The reference's own diagnostics 1 and 3: Monte Carlo estimate of $\nabla_\theta J^\lambda(\theta)$ (here, `reinforce`'s score-function estimator) vs. the exact oracle (`Portfolio.exact_gradient`), repeated across $\lambda\in\{0.025,0.05,0.1,0.2,0.4\}$. Unlike every discrete benchmark, the oracle here is exact, not another estimator — this isolates `reinforce`'s bias/variance exactly. `reinforce`'s bias mixes genuine $O(\lambda^2)$ perturbation bias with the estimator's own missing mean-field term (see the model markdown); the reference predicts growing MC variability as $\lambda\to0$.

In [ ]:
from mfc.algorithms.portfolio import reinforce_step

reps = 30
bias_norm, std_norm = {}, {}
gen = torch.Generator(device="cpu").manual_seed(0)
for lam in lambdas:
    theta_lam = by_alg_lambda[("reinforce", lam)]["theta_final"]
    oracle = env.exact_gradient(theta_lam, lam)
    theta_attached = theta_lam.clone().requires_grad_(True)
    samples = torch.stack([reinforce_step(env, theta_attached, lam, cfg.B, generator=gen) for _ in range(reps)])
    bias_norm[lam] = (samples.mean(dim=0) - oracle).norm().item()
    std_norm[lam] = samples.std(dim=0).norm().item()

fig, ax = viz.plot_horizon_scaling(bias_norm, xlabel="lambda", ylabel="norm over theta (T*2 components)", label="||bias||", integer_xaxis=False)
viz.plot_horizon_scaling(std_norm, xlabel="lambda", label="||std||", color_index=1, integer_xaxis=False, ax=ax)
ax.set_title("reinforce gradient estimator vs the exact gradient: bias/std norms vs lambda", loc="left")

## Robustness to non-Gaussian returns

The reference's own diagnostic 4: replacing the Gaussian excess-return law with a centered, variance-matched Student-$t_5$ (`PortfolioConfig(return_distribution="student_t")`, `Portfolio.sample_returns`) — same $\bar r,\sigma_R$, heavier tails. Since the exact objective/gradient only use the first two moments, $J^0(\theta^\star)$ is *unchanged*; what changes is the `reinforce` Monte Carlo estimator's sampling variability, which this compares directly.

In [ ]:
env_t = Portfolio(PortfolioConfig(return_distribution="student_t"), device="cpu")
assert env_t.exact_objective(theta_star, 0.0).item() == J_star  # moments-only oracle is exactly return-law-invariant

gen = torch.Generator(device="cpu").manual_seed(0)
samples_gauss = torch.stack([reinforce_step(env, theta_star.clone().requires_grad_(True), 0.1, cfg.B, generator=gen) for _ in range(reps)])
samples_t = torch.stack([reinforce_step(env_t, theta_star.clone().requires_grad_(True), 0.1, cfg.B, generator=gen) for _ in range(reps)])
print(f"reinforce gradient std norm, Gaussian returns:   {samples_gauss.std(dim=0).norm().item():.4f}")
print(f"reinforce gradient std norm, Student-t(5) returns: {samples_t.std(dim=0).norm().item():.4f}")

## Horizon scaling

Final validation objective and $\|\theta_{\text{learned}}-\theta^\star\|$ vs horizon $T$, using `runs/portfolio/main/` if it has been populated (`scripts/train.py --env portfolio --alg <alg> --config main`); otherwise this cell notes that main-tier data is needed and skips.

In [ ]:
main_runs_dir = ROOT / "runs" / "portfolio" / "main"
main_runs = [r for alg in cfg.algorithms if (main_runs_dir / f"{alg}_T5_lam0.1_seed0.pt").exists() for r in load_runs("portfolio", alg, "main")] if main_runs_dir.exists() else []

if not main_runs:
    print("no runs/portfolio/main/ data yet -- run scripts/train.py --env portfolio --alg exact_gradient --config main "
          "(and --alg reinforce) to populate this cell")
else:
    from configs.portfolio import MAIN
    final_J_by_T = {}
    theta_err_by_T = {}
    for T_h in MAIN.horizons:
        group = [r for r in main_runs if r["alg"] == "exact_gradient" and r["lam"] == 0.1 and r["T"] == T_h]
        theta_h_star = env.optimal_theta(T_h)
        final_J_by_T[T_h] = sum(r["validation_J"][-1].item() for r in group) / len(group)
        theta_err_by_T[T_h] = sum((r["theta_final"] - theta_h_star).norm().item() for r in group) / len(group)

    fig, ax = viz.plot_horizon_scaling(final_J_by_T, ylabel="final validation J (exact_gradient, lambda=0.1)", label="J")
    ax.set_title("Horizon scaling: final objective", loc="left")
    fig, ax = viz.plot_horizon_scaling(theta_err_by_T, ylabel="||theta_learned - theta*||", label="theta error", color_index=1)
    ax.set_title("Horizon scaling: theta error vs optimum", loc="left")

## Generalization without retraining

Evaluating the $\lambda=0.1$ `exact_gradient`-learned $\theta$ exactly (no retraining, no Monte Carlo) under a different initial wealth, a stronger perturbation intensity, and model misspecification (shifted risk-aversion / risky-asset premium). $\theta$ is horizon-specific (see `mfc.environments.lq`'s module docstring for why), so there is no "different $T$" scenario here.

In [ ]:
scenarios = {
    "baseline (mu0, rho)": env,
    "mu0 x 2 (wealthier start)": Portfolio(PortfolioConfig(mu0=env.config.mu0 * 2), device="cpu"),
    "mu0 = 0.5 (poorer start)": Portfolio(PortfolioConfig(mu0=0.5), device="cpu"),
    "rho x 2 (stronger perturbation)": Portfolio(PortfolioConfig(rho=env.config.rho * 2), device="cpu"),
    "2x risk aversion (chi)": Portfolio(PortfolioConfig(chi=env.config.chi * 2), device="cpu"),
    "2x risky-asset premium (r_bar)": Portfolio(PortfolioConfig(r_bar=env.config.r_bar * 2), device="cpu"),
}
gen_results = [{"name": name, "J": sc_env.exact_objective(theta_exact, 0.1)} for name, sc_env in scenarios.items()]
fig, ax = viz.plot_generalization(gen_results)
ax.set_title("J^0.1 under different scenarios (theta fixed at lambda=0.1's learned value, no retraining)", loc="left")

## Missing mean-field term: exact_gradient vs reinforce

`reinforce` treats the population mean as exogenous — the same ablation `mfc.algorithms.reinforce` implements for the discrete benchmarks and `mfc.algorithms.lq` for LQ (context.md: "Show missing mean-field term by comparison with reinforce"). Final validation objective for both algorithms, across seeds, at $\lambda=0.1$.

In [ ]:
for alg in cfg.algorithms:
    group = [r for r in runs if r["alg"] == alg and r["lam"] == 0.1]
    finals = torch.tensor([r["validation_J"][-1].item() for r in group])
    print(f"{alg:<14} final validation J: mean={finals.mean().item():.4f}  std={finals.std().item() if len(finals) > 1 else 0.0:.4f}  (J*={J_star:.4f})")

## Sample trajectories: learned vs optimal

One sampled $X_t$ (wealth) trajectory under the learned policy ($\lambda=0.1$, `exact_gradient`) and one under $\theta^\star$, both from $X_0\sim\mathcal N(\mu_0,\Sigma_0)$.

In [ ]:
gen_l = torch.Generator(device="cpu").manual_seed(0)
gen_o = torch.Generator(device="cpu").manual_seed(1)
learned_traj = env.rollout(theta_exact, lam=0.0, B=1, generator=gen_l)["X"][:, 0]
optimal_traj = env.rollout(theta_star, lam=0.0, B=1, generator=gen_o)["X"][:, 0]

fig, ax = viz.plot_trajectories(learned_traj, optimal_traj)
ax.set_ylabel("X_t (wealth)")
ax.set_title("Sample wealth trajectories (learned solid, optimal dashed)", loc="left")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")